# Testing Models

In [8]:
import pandas as pd
from taxipred.utils.constants import NEW_CSV_PATH

df = pd.read_csv(NEW_CSV_PATH)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 951 entries, 0 to 950
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   trip_distance_km       901 non-null    float64
 1   time_of_day            902 non-null    object 
 2   day_of_week            905 non-null    object 
 3   traffic_conditions     901 non-null    object 
 4   weather                905 non-null    object 
 5   base_fare              907 non-null    float64
 6   per_km_rate            907 non-null    float64
 7   per_minute_rate        902 non-null    float64
 8   trip_duration_minutes  905 non-null    float64
 9   trip_price             951 non-null    float64
dtypes: float64(6), object(4)
memory usage: 74.4+ KB


## Seperating features and target

In [9]:
X, y = df.drop(columns="trip_price"), df["trip_price"]
X.shape, y.shape

((951, 9), (951,))

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

## Imputation

- Imputing the NaN values with best performing methods based on our evaluations.

In [11]:
numeric_columns = X_train.select_dtypes(include=["number"]).columns
cat_columns = X_train.select_dtypes(include=["object"]).columns

numeric_columns, cat_columns

(Index(['trip_distance_km', 'base_fare', 'per_km_rate', 'per_minute_rate',
        'trip_duration_minutes'],
       dtype='object'),
 Index(['time_of_day', 'day_of_week', 'traffic_conditions', 'weather'], dtype='object'))

### Numerical Columns
- Using MICE (Multiple Imputation by Chained Equations) to impute null values for numerical columns the performance on our actual data.

In [12]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.linear_model import LinearRegression

X_train_num_imp = X_train[numeric_columns].copy()

mice_imputer = IterativeImputer(
    estimator=LinearRegression(),
    random_state=42,
    max_iter=20,
    sample_posterior=False
)

X_train_num_imp = pd.DataFrame(
    mice_imputer.fit_transform(X_train_num_imp),
    columns=X_train_num_imp.columns,
    index=X_train_num_imp.index
)

X_train_num_imp.info()

<class 'pandas.core.frame.DataFrame'>
Index: 637 entries, 181 to 102
Data columns (total 5 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   trip_distance_km       637 non-null    float64
 1   base_fare              637 non-null    float64
 2   per_km_rate            637 non-null    float64
 3   per_minute_rate        637 non-null    float64
 4   trip_duration_minutes  637 non-null    float64
dtypes: float64(5)
memory usage: 29.9 KB


In [13]:
X_test_num_imp = X_test[numeric_columns].copy()
X_test_num_imp = pd.DataFrame(
    mice_imputer.transform(X_test[numeric_columns]),
    columns=numeric_columns,
    index=X_test.index
)

In [19]:
X_train_num_imp.info(), X_test_num_imp.info()

<class 'pandas.core.frame.DataFrame'>
Index: 637 entries, 181 to 102
Data columns (total 5 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   trip_distance_km       637 non-null    float64
 1   base_fare              637 non-null    float64
 2   per_km_rate            637 non-null    float64
 3   per_minute_rate        637 non-null    float64
 4   trip_duration_minutes  637 non-null    float64
dtypes: float64(5)
memory usage: 29.9 KB
<class 'pandas.core.frame.DataFrame'>
Index: 314 entries, 199 to 891
Data columns (total 5 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   trip_distance_km       314 non-null    float64
 1   base_fare              314 non-null    float64
 2   per_km_rate            314 non-null    float64
 3   per_minute_rate        314 non-null    float64
 4   trip_duration_minutes  314 non-null    float64
dtypes: float64(5)
memory usa

(None, None)

### Categorical Columns
- Most-frequent imputation will be used for categorical variables because it provides stable and consistently strong performance across categories while avoiding unnecessary model complexity

In [22]:
X_train_cat_imp = X_train[cat_columns].copy()
X_test_cat_imp = X_test[cat_columns].copy()

In [23]:
cat_imputer = SimpleImputer(strategy="most_frequent")


X_train_cat_imp = pd.DataFrame(
    cat_imputer.fit_transform(X_train_cat_imp),
    columns=cat_columns,
    index=X_train.index
)


In [24]:
X_test_cat_imp = pd.DataFrame(
    cat_imputer.transform(X_test_cat_imp),
    columns=cat_columns,
    index=X_test.index
)